<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day4/ExerciseXP/Exercises_XP_Day4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [3]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [4]:
# Exercise 1: Implement `LoRALayer`

# Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # Application de l'adaptation de faible rang : x @ A @ B multiplié par le facteur d'échelle alpha
        # Forme de x : (batch, in_dim) -> Forme finale : (batch, out_dim)
        x = (x @ self.A @ self.B) * self.alpha
        return x

# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 10     # Dimension d'entrée arbitraire pour le test
out_dim = 5     # Dimension de sortie arbitraire pour le test
rank = 2        # Rang faible r (typiquement inférieur à in_dim ou out_dim)
alpha = 1.0     # Facteur d'échelle hyperparamétrique (scaling factor)

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(2, in_dim)  # Génération d'un tenseur d'entrée factice avec un batch size de 2

print("Tenseur d'entrée x :\n", x)
print("Couche LoRALayer :\n", layer)
print("Original output:\n", layer(x))


Tenseur d'entrée x :
 tensor([[-0.3885, -0.9343, -0.4991, -1.0867, -0.2044, -2.2685, -0.9133, -0.4204,
          1.3111, -0.2199],
        [ 0.2190,  0.2045,  0.6177, -0.2876,  0.8218,  0.1512,  0.1036, -2.1996,
         -0.0885, -0.5612]])
Couche LoRALayer :
 LoRALayer()
Original output:
 tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [5]:
# Exercise 2: Wrap `nn.Linear` with LoRA

# Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # Additionne les sorties de la projection linéaire standard et du module adaptatif LoRA
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
# Encapsulation de notre couche de base avec les paramètres du sandbox (rank=2, alpha=1.0)
layer_lora_1 = LinearWithLoRA(base_linear, rank=rank, alpha=alpha)
print("LinearWithLoRA output:\n", layer_lora_1(x))


LinearWithLoRA output:
 tensor([[ 1.3557, -1.0487, -0.3321, -0.3467, -0.3075],
        [-0.1582, -0.6605,  0.8469, -0.0565, -0.7080]], grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [6]:
# Exercise 3: Swap a simple network layer with LoRA

# Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`.
# The outputs should match before training because the LoRA adapters start at zero.

class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

# Assignation des hyperparamètres du bac à sable (in_dim=10, out_dim=5)
single_net = SingleLayerNet(num_features=in_dim, num_classes=out_dim)
sample_input = torch.randn(2, in_dim)  # Tenseur d'entrée avec batch_size de 2

with torch.no_grad():
    baseline_output = single_net(sample_input)

# Remplacement dynamique de la couche linéaire classique par notre wrapper LoRA (rank=2, alpha=1.0)
single_net.layer = LinearWithLoRA(single_net.layer, rank=rank, alpha=alpha)

with torch.no_grad():
    lora_output = single_net(sample_input)

# Vérification de l'égalité absolue à l'aide de la fonction d'équivalence de PyTorch
is_matching = torch.allclose(baseline_output, lora_output)
print("Outputs match before training?", is_matching)


Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [7]:
# Exercise 4: Merged-weight LoRA layer

# Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B
        # Fusion mathématique : poids d'origine + (alpha * Delta_W transposé)
        # La transposition .T est requise car PyTorch stocke les poids d'une couche linéaire sous la forme (out_features, in_features)
        combined_weight = self.linear.weight + self.lora.alpha * lora.T
        return F.linear(x, combined_weight, self.linear.bias)

# Instanciation de la couche fusionnée à l'aide de notre couche de base et des hyperparamètres du bac à sable
layer_lora_2 = LinearWithLoRAMerged(base_linear, rank=rank, alpha=alpha)
print("Merged LoRA output:\n", layer_lora_2(x))


Merged LoRA output:
 tensor([[ 1.3557, -1.0487, -0.3321, -0.3467, -0.3075],
        [-0.1582, -0.6605,  0.8469, -0.0565, -0.7080]],
       grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [8]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            # Aplatit les images MNIST 2D (batch, 1, 28, 28) en vecteurs 1D (batch, 784)
            nn.Flatten(),
            nn.Linear(num_features, num_hidden_1), # Première couche linéaire
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2), # Deuxième couche linéaire
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),  # Couche de sortie finale
        )

    def forward(self, x):
        x = self.layers(x)
        return x


In [9]:
# Architecture
num_features = 28 * 28  # 784 pixels en entrée
num_hidden_1 = 128      # Taille de la première couche cachée
num_hidden_2 = 64       # Taille de la deuxième couche cachée
num_classes = 10        # 10 classes pour les chiffres de 0 à 9

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.005
num_epochs = 2          # Époques courtes pour une exécution rapide en lab

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Device utilisé :", DEVICE)
print(model)
print(optimizer_pretrained)


Device utilisé : cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [10]:
BATCH_SIZE = 64

# Chargement de l'ensemble d'entraînement
train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

# 1. Chargement de l'ensemble de test (train=False)
test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

# 2. Création du DataLoader d'entraînement (avec mélange/shuffle actif)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 3. Création du DataLoader de test (mélange/shuffle inutile)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Boucle de vérification des dimensions du premier lot
for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)  # Doit afficher torch.Size([64, 1, 28, 28])
    print('Image label dimensions:', labels.shape)  # Doit afficher torch.Size([64])
    break


100%|██████████| 9.91M/9.91M [00:00<00:00, 149MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 35.2MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 73.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.19MB/s]


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [12]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            # Transfert des tenseurs sur le périphérique matériel actif (GPU ou CPU)
            features = features.to(device)
            targets = targets.to(device)

            # Passage avant (Forward pass) pour obtenir les logits
            logits = model(features)

            # Extraction des étiquettes prédites avec la valeur maximale de logit
            _, predicted_labels = torch.max(logits, 1)

            # Accumulation du nombre total d'exemples et des prédictions correctes
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()

    # Retourne le score final sous forme de pourcentage (0.0 à 100.0)
    return (correct_pred.float() / num_examples) * 100


## Training

In [13]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            # 1. Déplacement sécurisé des caractéristiques et des étiquettes sur le périphérique (GPU/CPU)
            features = features.to(device)
            targets = targets.to(device)

            # 2. Passe avant (Forward pass) pour obtenir les logits
            logits = model(features)

            # 3. Calcul de la perte de classification multiclasse (Cross Entropy)
            loss = F.cross_entropy(logits, targets)

            # 4. Réinitialisation des gradients accumulés au tour précédent
            optimizer.zero_grad()

            # 5. Rétropropagation de l'erreur (Backward pass)
            loss.backward()

            # 6. Mise à jour des poids du modèle par l'optimiseur
            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss.item()))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))


In [14]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 001/002|Batch 000/938| Loss: 2.3160
Epoch: 001/002|Batch 400/938| Loss: 0.0598
Epoch: 001/002|Batch 800/938| Loss: 0.0955
Epoch: 001/002 training accuracy: 95.77%
Time elapsed: 0.29 min
Epoch: 002/002|Batch 000/938| Loss: 0.1870
Epoch: 002/002|Batch 400/938| Loss: 0.0682
Epoch: 002/002|Batch 800/938| Loss: 0.0339
Epoch: 002/002 training accuracy: 97.86%
Time elapsed: 0.57 min
Total Training Time: 0.57 min
Test accuracy: 96.90%


# Replacing Linear with LoRA Layers

In [15]:
model_lora = copy.deepcopy(model)

# Remplacement des couches linéaires (indices 1, 3 et 5) par notre version fusionnée LoRA
model_lora.layers[1] = LinearWithLoRAMerged(model_lora.layers[1], rank=4, alpha=8)
model_lora.layers[3] = LinearWithLoRAMerged(model_lora.layers[3], rank=4, alpha=8)
model_lora.layers[5] = LinearWithLoRAMerged(model_lora.layers[5], rank=4, alpha=8)

model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')


MultilayerPerceptron(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (2): ReLU()
    (3): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (4): ReLU()
    (5): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model:96.90%
Test accuracy LoRA model:96.90%


## Freezing the Original Linear Layers

In [16]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.1.linear.weight:False
layers.1.linear.bias:False
layers.1.lora.A:True
layers.1.lora.B:True
layers.3.linear.weight:False
layers.3.linear.bias:False
layers.3.lora.A:True
layers.3.lora.B:True
layers.5.linear.weight:False
layers.5.linear.bias:False
layers.5.lora.A:True
layers.5.lora.B:True


In [17]:
# Filtrer les paramètres pour n'entraîner EXCLUSIVEMENT que les adaptateurs LoRA actifs
optimizer_lora = torch.optim.Adam(filter(lambda p: p.requires_grad, model_lora.parameters()), lr=learning_rate)

# Lancement de l'ajustement fin LoRA
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)

print("\n" + "="*60)
print(f'🏆 Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')
print("="*60)

print(f'Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')


Epoch: 001/002|Batch 000/938| Loss: 0.1200
Epoch: 001/002|Batch 400/938| Loss: 0.2455
Epoch: 001/002|Batch 800/938| Loss: 0.0670
Epoch: 001/002 training accuracy: 97.51%
Time elapsed: 0.29 min
Epoch: 002/002|Batch 000/938| Loss: 0.1845
Epoch: 002/002|Batch 400/938| Loss: 0.1263
Epoch: 002/002|Batch 800/938| Loss: 0.1769
Epoch: 002/002 training accuracy: 97.93%
Time elapsed: 0.58 min
Total Training Time: 0.58 min

🏆 Test accuracy LoRA finetune: 97.01%
Test accuracy orig model: 96.90%
Test accuracy LoRA model: 97.01%
